# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring a FAIR<sup>2</sup>-certified dataset using the `mlcroissant` library, with complete referencing by each entity's `@id` as per best practice.

### Dataset Source
The dataset is described by a Croissant schema available from the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and initialize the dataset for exploration. The metadata includes the title, description, and additional information, all accessible programmatically with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a mlcroissant.internal.structure.Metadata object
# Display some high-level metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")


## 2. Data Overview
Explore available **record sets** and their fields, referencing each by their `@id`. These `@id`s are required for all data extraction and manipulation in subsequent steps.

Let's inspect the record sets defined in the dataset and their associated fields. This helps us know what data (tables/columns) are available and how to reference each for programmatic access.

In [ ]:
# List all record sets by `@id` and display their included fields/columns by @id

record_sets = list(dataset.record_sets)

print("Available record sets and their fields/columns:")

for rs in record_sets:
    print(f"Record Set: {rs.id}")
    try:
        # For each field (these may be fields or columns depending on the Croissant version used)
        field_ids = [f.id for f in getattr(rs, 'fields', [])] or [c.id for c in getattr(rs, 'columns', [])]
        print(f"  Fields/Columns (@id): {field_ids}")
    except Exception as e:
        print(f"  Could not extract fields for this record set: {e}")
    print("-")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. We use the record set and field `@id`s from the overview above. Note: All data entities are referenced by their `@id` for robustness and reproducibility.

In [ ]:
# For demonstration, extract each record set into a DataFrame using its @id

dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records from record set '@id': {rs_id}")
    if not df.empty:
        print(f"Columns (fields/columns by @id): {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found in record set '@id': {rs_id}.")
    print("-")
    dataframes[rs_id] = df


## 4. Exploratory Data Analysis (EDA)
Apply processing to the main record set (table) of the dataset. Select a numeric field (by its `@id` as seen above), filter, normalize, and group by a categorical field for deeper analysis. All field access and references use their `@id`.

In [ ]:
# Example: Select the first non-empty record set for EDA

# Find the main record set (with the most records)
main_rs_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_rs_id = rsid
        break
if main_rs_id is None:
    raise RuntimeError("No non-empty record set found for EDA.")

main_df = dataframes[main_rs_id]
print(f"Selected record set for EDA: {main_rs_id}")
print(f"Available fields: {main_df.columns.tolist()}")

# Attempt to automatically select a numeric field for demo (prefer GAD-7/PHQ-9/PSQ scores if available)
preferred_numeric_fields = [c for c in main_df.columns if any(x in c.lower() for x in ['gad', 'phq', 'psq', 'score', 'age'])]
numeric_field = None
if preferred_numeric_fields:
    numeric_field = preferred_numeric_fields[0]
else:
    # fallback: pick first numeric column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            break
if numeric_field is None:
    raise ValueError("No numeric field found for EDA.")

print(f"Selected numeric field for filtering/normalization: {numeric_field}")
# Remove missing values for numeric analysis
valid_df = main_df[main_df[numeric_field].notnull()]

threshold = valid_df[numeric_field].quantile(0.75)  # 75th percentile as example threshold

filtered_df = valid_df[valid_df[numeric_field] > threshold].copy()
print(f"\nFiltered records where {numeric_field} > {threshold:.2f} (75th percentile):")
print(filtered_df[[numeric_field]].head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nTop 5 (normalized):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a categorical field (prefer 'gender' or similar, by @id)
preferred_group_fields = [c for c in main_df.columns if any(x in c.lower() for x in ['gender', 'sex', 'village', 'education'])]
group_field = preferred_group_fields[0] if preferred_group_fields else None
if group_field and group_field in filtered_df.columns:
    print(f"\nGrouping by: {group_field}")
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped)
else:
    print("No suitable group field found for grouping in EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and (if possible) the grouped means by a key attribute (e.g., gender or village).

In [ ]:
# Inline visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field} (@id)')
plt.xlabel(numeric_field + ' (@id)')
plt.ylabel('Count')
plt.show()

# If a group_field was found above, visualize group means
if 'grouped' in locals():
    grouped.plot(kind='bar', color='tab:blue', edgecolor='k')
    plt.title(f'Mean {numeric_field} by {group_field} (@id)')
    plt.xlabel(group_field + ' (@id)')
    plt.ylabel(f'Mean {numeric_field}')
    plt.tight_layout()
    plt.show()


## 6. Conclusion

- Demonstrated programmatic access to a Croissant metadata-driven dataset using `mlcroissant`, referencing all data entities by their `@id` for clarity and reproducibility.
- Outlined the available record sets and fields; loaded one as DataFrame and performed basic exploratory filtering, normalization, grouping, and visualization.
- This notebook is a starting point for deeper analysis of mental health survey data for Kilifi County, Kenya, including custom feature engineering, statistical analysis, or ML modeling following the same referencing principles.

For further analysis: explore handling of missing values, merge with external data, or apply domain-specific transformations using the persistent Croissant ids.